### In this notebook, we woll create a basic Q&A chain and Agent over a SQL database

#### Architecture
At a high-level, the steps of any SQL chain and agent are:
- Convert question to SQL query using the LLM model
- Execute the SQL query
- Getting the model to respond to user input using the query result

In [1]:
from langchain.utilities import SQLDatabase
from langchain_google_genai import GoogleGenerativeAI
from langchain_experimental.sql import SQLDatabaseChain
from langchain.prompts import PromptTemplate
from langchain.prompts.chat import HumanMessagePromptTemplate
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI
from langchain.schema import HumanMessage, SystemMessage
from langchain.chains import create_sql_query_chain
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.0)

In [6]:
host = "localhost"
port = 3306
username = "test"
password = "XXXXX"
database_schema = "analytics"
mysql_uri = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database_schema}"

db= SQLDatabase.from_uri(mysql_uri, sample_rows_in_table_info=5)
chain = create_sql_query_chain(llm,db)

In [7]:
print(db.dialect)
print(db.get_usable_table_names())

mysql
['customer', 'employee', 'manufacturer', 'products', 'sales']


In [12]:
response = chain.invoke({"question":"How many cutomers are there in the customers table?"})
response

'SQLQuery: SELECT COUNT(*) AS `customer_count` FROM `customer`;'

In [16]:
db_query= response.split("SQLQuery:")
db_query[1]

' SELECT COUNT(*) AS `customer_count` FROM `customer`;'

In [17]:
db.run(db_query[1])

'[(10,)]'

In [18]:
response1 = chain.invoke({"question":"Which is the most sold product"})
response1

'```sql\nSELECT\n  `name`\nFROM products\nWHERE\n  `product_id` = (\n    SELECT\n      `product_id`\n    FROM sales\n    GROUP BY\n      `product_id`\n    ORDER BY\n      SUM(`quantity`) DESC\n    LIMIT 1\n  );\n```'

In [22]:
clean_query = response1.replace("sql", "").replace("", "").strip() 
clean_query = clean_query.replace("```","")
print(clean_query)


SELECT
  `name`
FROM products
WHERE
  `product_id` = (
    SELECT
      `product_id`
    FROM sales
    GROUP BY
      `product_id`
    ORDER BY
      SUM(`quantity`) DESC
    LIMIT 1
  );



In [23]:
db.run(clean_query)

"[('Instrument',)]"